
# Transformações Geométricas 3D com Coordenadas Homogêneas

Este notebook apresenta a teoria essencial e implementa **transformações 3D** usando **matrizes 4×4** em coordenadas homogêneas, com operações **manuais** em Python, sem uso de NumPy.

## Visão geral teórica

**Coordenadas homogêneas**: um ponto 3D cartesiano $(x, y, z)$ é representado como $(x, y, z, 1)$. Vetores direção usam $w = 0$.  
A vantagem é representar **todas as transformações afins** como multiplicação de matriz.

**Matriz 4×4 geral:**

$$
M =
\begin{bmatrix}
r_{11} & r_{12} & r_{13} & t_x \\
r_{21} & r_{22} & r_{23} & t_y \\
r_{31} & r_{32} & r_{33} & t_z \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

onde $R = [r_{ij}]$ é a parte linear e $\mathbf{t} = (t_x, t_y, t_z)$ é a translação.

### Transformações básicas

1. **Identidade**  
$ I_4 $ não altera ponto algum.

2. **Translação** por $(t_x, t_y, t_z)$:  
$$
T =
\begin{bmatrix}
1 & 0 & 0 & t_x \\
0 & 1 & 0 & t_y \\
0 & 0 & 1 & t_z \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

3. **Escala** por $(s_x, s_y, s_z)$:  
$$
S =
\begin{bmatrix}
s_x & 0 & 0 & 0 \\
0 & s_y & 0 & 0 \\
0 & 0 & s_z & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

4. **Rotações** em torno dos eixos principais, ângulo $	heta$ em radianos:

- Em **X**:  
$$
R_x(\theta) =
\begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & \cos\theta & -\sin\theta & 0 \\
0 & \sin\theta & \cos\theta & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

- Em **Y**:  
$$
R_y(\theta) =
\begin{bmatrix}
\cos\theta & 0 & \sin\theta & 0 \\
0 & 1 & 0 & 0 \\
-\sin\theta & 0 & \cos\theta & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

- Em **Z**:  
$$
R_z(\theta) =
\begin{bmatrix}
\cos\theta & -\sin\theta & 0 & 0 \\
\sin\theta & \cos\theta & 0 & 0 \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

5. **Reflexões** em planos coordenados, por exemplo no plano **XY**:  
$$
\text{Ref}_{XY} =
\begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & -1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

6. **Cisalhamentos** opcionais, por exemplo cisalhar X em função de Y e Z:  
$$
\text{Shear}_X(k_y, k_z) =
\begin{bmatrix}
1 & k_y & k_z & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

### Composição e ordem
Aplicar múltiplas transformações corresponde a **multiplicar as matrizes** na ordem correta.  
Se o vetor coluna $p$ é pós-multiplicado, e você quer **aplicar A depois B**, então:

$$
p' = B(Ap) = (BA)p
$$

Logo, a matriz mais à **direita** atua **primeiro**.

### Sistemas de referência
- **Objeto**: coordenadas locais para modelagem.  
- **Mundo**: posiciona o objeto na cena.  
- **Visualização**: transforma para o sistema da câmera.  



In [ ]:

from math import cos, sin, radians

def identity4():
    return [
        [1.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0],
    ]

def matmul(A, B):
    # 4x4 * 4x4 -> 4x4
    C = [[0.0]*4 for _ in range(4)]
    for i in range(4):
        for j in range(4):
            s = 0.0
            for k in range(4):
                s += A[i][k]*B[k][j]
            C[i][j] = s
    return C

def matvec(M, v):
    # 4x4 * 4 -> 4
    assert len(v) == 4
    out = [0.0]*4
    for i in range(4):
        s = 0.0
        for j in range(4):
            s += M[i][j]*v[j]
        out[i] = s
    return out

def translate(tx, ty, tz):
    T = identity4()
    T[0][3] = float(tx)
    T[1][3] = float(ty)
    T[2][3] = float(tz)
    return T

def scale(sx, sy, sz):
    S = identity4()
    S[0][0] = float(sx)
    S[1][1] = float(sy)
    S[2][2] = float(sz)
    return S

def rotate_x(theta_deg):
    t = radians(theta_deg)
    c, s = cos(t), sin(t)
    R = identity4()
    R[1][1] = c;  R[1][2] = -s
    R[2][1] = s;  R[2][2] =  c
    return R

def rotate_y(theta_deg):
    t = radians(theta_deg)
    c, s = cos(t), sin(t)
    R = identity4()
    R[0][0] =  c; R[0][2] =  s
    R[2][0] = -s; R[2][2] =  c
    return R

def rotate_z(theta_deg):
    t = radians(theta_deg)
    c, s = cos(t), sin(t)
    R = identity4()
    R[0][0] =  c; R[0][1] = -s
    R[1][0] =  s; R[1][1] =  c
    return R

def reflect_xy():
    R = identity4()
    R[2][2] = -1.0
    return R

def shear_x(ky=0.0, kz=0.0):
    Sh = identity4()
    Sh[0][1] = float(ky)
    Sh[0][2] = float(kz)
    return Sh

def to_point(x, y, z):
    return [float(x), float(y), float(z), 1.0]

def to_vector(x, y, z):
    return [float(x), float(y), float(z), 0.0]

def apply_transform(M, points):
    return [matvec(M, p) for p in points]

def homogenize(v):
    # Converts homogeneous [x, y, z, w] to Cartesian (dividing by w if needed)
    x, y, z, w = v
    if w != 0.0:
        return [x/w, y/w, z/w]
    return [x, y, z]  # for direction vectors



## Exemplo 1. Ordem das transformações

Aplicar sequência: rotacionar em Z por 90 graus, depois transladar em \((2, 0, 0)\).  
Como pós-multiplicamos vetores, o ponto é multiplicado primeiro por \(R_z\) e depois por \(T\).  
A matriz composta correta é \(M = T \cdot R_z\).


In [ ]:

# Ponto inicial
p = to_point(1, 0, 0)

Rz = rotate_z(90)
T  = translate(2, 0, 0)

M = matmul(T, Rz)   # aplica Rz depois T
p_rot = matvec(Rz, p)
p_final = matvec(M, p)

print("p_rot:", homogenize(p_rot))     # deve ser ~ (0, 1, 0)
print("p_final:", homogenize(p_final)) # depois translada -> (2, 1, 0)


p_rot: [6.123233995736766e-17, 1.0, 0.0]
p_final: [2.0, 1.0, 0.0]



## Exemplo 2. Transformando os vértices de um cubo

Construir um cubo unitário centrado na origem, aplicar escala não uniforme, rotação em Y e uma translação.


In [ ]:

# Vértices de um cubo centrado na origem, aresta 2
cube = []
for x in (-1, 1):
    for y in (-1, 1):
        for z in (-1, 1):
            cube.append(to_point(x, y, z))

S  = scale(1.0, 2.0, 0.5)
Ry = rotate_y(30)
T  = translate(3, -1, 2)

# Ordem correta: primeiro S, depois Ry, depois T  ->  M = T * Ry * S
M = matmul(T, matmul(Ry, S))

cube_transformed = [homogenize(matvec(M, p)) for p in cube]
for i, v in enumerate(cube_transformed, 1):
    print(f"V{i}:", [round(c, 4) for c in v])


V1: [1.884, -3.0, 2.067]
V2: [2.384, -3.0, 2.933]
V3: [1.884, 1.0, 2.067]
V4: [2.384, 1.0, 2.933]
V5: [3.616, -3.0, 1.067]
V6: [4.116, -3.0, 1.933]
V7: [3.616, 1.0, 1.067]
V8: [4.116, 1.0, 1.933]



## Exemplo 3. Reflexão e cisalhamento

Refletir no plano XY e, em seguida, cisalhar X em função de Y e Z.


In [ ]:

p = to_point(2, 3, 4)
Ref = reflect_xy()
Sh  = shear_x(ky=0.2, kz=-0.1)

M = matmul(Sh, Ref)  # Ref primeiro, depois Sh
out = homogenize(matvec(M, p))
print("Resultado:", [round(c, 4) for c in out])


Resultado: [3.0, 3.0, -4.0]
